In [1]:
import pandas as pd

train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch6/diabetes_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch6/diabetes_test.csv")

print("===== 데이터 크기 =====")
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
print("\n")

print("===== 데이터 정보(자료형) =====")
print(train.info())
print("\n")

print("===== train 결측치 수 =====")
print(train.isnull().sum().sum())
print("\n")

print("===== test 결측치 수 =====")
print(train.isnull().sum().sum())
print("\n")

print("===== target 빈도 =====")
print(train["Outcome"].value_counts())

target = train.pop("Outcome")

from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

print("\n===== 분할된 데이터 크기 =====")
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)

from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict_proba(X_val)

print("\n===== 예측 결과 확인 (샘플 5개) =====")
print(pred[:5])

from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(y_val, pred[:,1])
print('\nroc_auc:', roc_auc)

pred = rf.predict_proba(test)
submit = pd.DataFrame({"pred":pred[:,1]})
submit.to_csv("result.csv", index=False)

print("\n===== 제출파일 (샘플 5개) =====")
print(pd.read_csv("result.csv").head())

===== 데이터 크기 =====
Train Shape: (614, 9)
Test Shape: (154, 8)


===== 데이터 정보(자료형) =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               614 non-null    int64  
 1   Glucose                   614 non-null    int64  
 2   BloodPressure             614 non-null    int64  
 3   SkinThickness             614 non-null    int64  
 4   Insulin                   614 non-null    int64  
 5   BMI                       614 non-null    float64
 6   DiabetesPedigreeFunction  614 non-null    float64
 7   Age                       614 non-null    int64  
 8   Outcome                   614 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 43.3 KB
None


===== train 결측치 수 =====
0


===== test 결측치 수 =====
0


===== target 빈도 =====
Outcome
0    403
1    211
Name: count, dtype: int64

===== 분할된 데이터 크

---

### 1. 탐색적 데이터 분석(EDA)이 필수적인 이유 (핵심 요약)



* **데이터의 '생김새' 파악**: `df.info()`나 `df.dtypes`를 통해 숫자형인 줄 알았던 컬럼이 문자열(Object)로 되어 있는 등의 문제를 사전에 발견합니다.
* **이상치(Outlier)와 결측치 처리 전략 수립**: 결측치가 너무 많으면 컬럼 자체를 버릴지(`dropna`), 혹은 중앙값으로 채울지(`fillna`)를 EDA 결과에 따라 결정합니다.
* **타겟 변수의 불균형 확인**: `value_counts()`를 통해 분류 문제에서 특정 클래스가 너무 적지는 않은지 확인하여, 평가 지표(Accuracy vs F1-score)를 선택하는 기준을 세웁니다.
* **변수 선택(Feature Selection)**: 상관관계(`corr()`) 분석을 통해 타겟 값과 관련이 높은 변수를 우선적으로 모델에 학습시킵니다.

---

### 2. EDA 초간단 Template 활용 팁 (CBT 시험용)

시험장에서는 시간이 촉박하므로, 작성해주신 템플릿을 **최대한 빠르게 실행해서 인사이트만 뽑아내는 것**이 관건입니다.

#### ① 결측치 및 자료형 확인
```python
# 결측치 합계 확인 (내림차순 정렬로 많은 것부터 확인)
print(df.isnull().sum().sort_values(ascending=False))

# 데이터 타입 확인 (수치형/범주형 분리 처리를 위해 필수)
print(df.info())
```

#### ② 타겟 변수의 분포 (Type II의 핵심)
```python
# 분류 문제일 경우: 클래스 불균형 확인
print(df[TARGET].value_counts(normalize=True))

# 회귀 문제일 경우: 수치 분포 확인
print(df[TARGET].describe())
```

#### ③ 상관관계 분석 (시각화 제약 고려)
> **주의**: 실제 빅데이터분석기사 실기 시험 환경(구름 IDE 등)에서는 **그래프 시각화(sns, plt)가 불가능하거나 확인하기 어려울 수 있습니다.** 따라서 시각화보다는 **수치 데이터(`corr()`)**를 보고 판단하는 연습이 더 중요합니다.

```python
# 수치형 변수 간의 상관계수만 출력하여 확인
print(df.corr(numeric_only=True)[TARGET].sort_values(ascending=False))
```

---

### 3. EDA 결과와 다음 단계의 연결 (예시)

| EDA 발견 사항 | 후속 조치 (전처리 및 모델링) |
| :--- | :--- |
| 특정 컬럼에 결측치가 50% 이상 | 해당 컬럼 제거 (`drop`) |
| 수치 데이터의 스케일 차이가 큼 | `StandardScaler` 또는 `MinMaxScaler` 적용 |
| 타겟 값이 한쪽으로 치우침 (9:1) | 평가 지표를 `roc_auc_score`나 `f1_score`로 설정 |
| 범주형 변수의 고윳값(unique)이 너무 많음 | 원-핫 인코딩 대신 라벨 인코딩 고려 |

---